# Session 12: Simulating ZeRO across 32 virtual ranks

I use one small PyTorch model to make the training workload concrete, then account for where parameters, gradients, and Adam's two moment tensors would live under ZeRO-0 through ZeRO-3. The ranks are logical CPU-side bookkeeping, not 32 GPUs or 32 worker processes. This notebook demonstrates the memory/communication trade-off; it cannot measure multi-GPU speedup.

## What runs, and what is simulated?

The demo is a `1 -> 64 -> 64 -> 1` tanh MLP trained for 20 Adam updates on a fixed 128-example synthetic regression batch. Each of the 32 virtual ranks receives four examples and computes its own gradient on CPU. The simulator averages full gradients for ZeRO-0/1, reduce-scatters gradient shards for ZeRO-2/3, and maintains optimizer moments in the corresponding replicated or sharded layout. The ranks run sequentially in one process, not on 32 GPUs or over a network, so matching losses validate the algorithmic bookkeeping—not multi-GPU speed.

There are 32 virtual ranks. The demo uses Adam with learning rate `0.001`, `beta1=0.9`, `beta2=0.999`, and `eps=1e-8`. For model-state estimates I count FP32 parameters, one FP32 gradient per parameter, and two FP32 Adam states (`m` and `v`). Collective communication volumes are ring-model estimates; the simulator does not send network messages.

In [ ]:
from pathlib import Path
import sys
import importlib
import torch

if (Path.cwd() / "Assignments/S12/zero_simulation.py").exists():
    repository_root = Path.cwd()
elif (Path.cwd() / "zero_simulation.py").exists() and Path.cwd().name == "S12":
    repository_root = Path.cwd().parents[1]
else:
    raise FileNotFoundError("Open this notebook from the repository root or Assignments/S12")

sys.path.insert(0, str(repository_root))
import Assignments.S12.zero_simulation as zero_simulation
zero_simulation = importlib.reload(zero_simulation)
from Assignments.S12.zero_simulation import (
    FLOAT32_BYTES,
    LOCAL_BATCH_SIZE,
    STAGES,
    WORLD_SIZE,
    build_comparison,
    make_plots,
    partition_counts,
    run_demo_training,
)
from IPython.display import display
import pandas as pd

torch_parameter_count, training_results = run_demo_training()
print(f"Virtual ranks: {WORLD_SIZE}; demo model parameters: {torch_parameter_count:,}")
display(pd.DataFrame.from_dict(training_results, orient="index").rename_axis("configuration"))

## Memory accounting: what gets sharded?

Let `P` be the number of parameters, `b=4` bytes per FP32 value, and `N=32` ranks. The baseline keeps `P` parameters, `P` gradients, and `2P` Adam values on every rank, or `4Pb` bytes per rank. ZeRO-1 shards only the optimizer values; ZeRO-2 also shards gradients; ZeRO-3 shards parameters as well. Removing replicated state also reduces **aggregate** storage: ZeRO-0 stores `4NPb`, ZeRO-1 stores `(2N+2)Pb`, ZeRO-2 stores `(N+3)Pb`, and ZeRO-3 stores `4Pb` across the ranks.

I distribute integer tensor elements as evenly as possible. Since the parameter count may not divide by 32, the per-rank column reports the busiest rank (ceiling-sized shard); aggregate state sums remain exact. The activation number is an additional rough estimate for two hidden activations and one output tensor at the local batch size. It excludes activation gradients, temporary buffers, communication buffers, allocator overhead, and ZeRO-3's transient gathered parameters.

In [ ]:
comparison = build_comparison(torch_parameter_count)
assert comparison.iloc[0]["persistent_state_bytes_aggregate"] == WORLD_SIZE * 4 * torch_parameter_count * FLOAT32_BYTES
assert comparison.iloc[-1]["persistent_state_bytes_aggregate"] == 4 * torch_parameter_count * FLOAT32_BYTES
shards = partition_counts(torch_parameter_count)
assert len(shards) == WORLD_SIZE and sum(shards) == torch_parameter_count
assert max(shards) - min(shards) <= 1
print(f"Rank-0 parameter shard: {shards[0]:,}; shard sizes range {min(shards):,}–{max(shards):,}")
print(f"FP32 bytes/value: {FLOAT32_BYTES}; local examples/rank: {LOCAL_BATCH_SIZE}")

memory_view = comparison[[
    "stage", "parameters_per_rank", "gradients_per_rank", "optimizer_values_per_rank",
    "persistent_state_bytes_per_rank", "activation_estimate_bytes_per_rank",
    "estimated_total_bytes_per_rank", "persistent_state_bytes_aggregate",
    "estimated_total_bytes_aggregate",
]].copy()
for column in memory_view.columns[4:]:
    memory_view[column] = (memory_view[column] / 1024**2).round(6)
memory_view = memory_view.rename(columns={
    "persistent_state_bytes_per_rank": "persistent MiB / busiest rank",
    "activation_estimate_bytes_per_rank": "activation estimate MiB / rank",
    "estimated_total_bytes_per_rank": "total estimate MiB / rank",
    "persistent_state_bytes_aggregate": "persistent MiB / all ranks",
"estimated_total_bytes_aggregate": "total estimate MiB / all ranks",
})
display(memory_view)

## Computation and communication

The matrix multiplications still do the same arithmetic in each data-parallel rank. I estimate training work as `6 * dense_weight_parameters * local_batch`: roughly two FLOPs per multiply-accumulate in the forward pass and about four more for backward. This ignores nonlinearities, bias work, optimizer arithmetic, and kernel overhead. Aggregate FLOPs are 32 times the per-rank estimate because all 32 ranks process distinct local examples.

For communication I use a simple ring model and count bytes sent per rank: all-reduce is `2(N-1)/N` payloads; reduce-scatter and all-gather are each `(N-1)/N` payloads. ZeRO-1 needs gradient all-reduce and gathers updated parameter shards; ZeRO-2 uses gradient reduce-scatter and gathers updated parameters; the simplified ZeRO-3 estimate gathers parameters before both forward and backward, then reduce-scatters gradients. Real implementations overlap collectives and gather by layer, so this is a comparison model, not a network benchmark.

In [ ]:
compute_comm_view = comparison[[
    "stage", "compute_flops_per_rank_per_step", "compute_flops_aggregate_per_step",
    "communication", "communication_bytes_sent_per_rank",
    "communication_bytes_sent_aggregate",
]].copy()
compute_comm_view["compute GFLOPs/rank/step"] = (
    compute_comm_view.pop("compute_flops_per_rank_per_step") / 1e9
).round(8)
compute_comm_view["aggregate GFLOPs/step"] = (
    compute_comm_view.pop("compute_flops_aggregate_per_step") / 1e9
).round(8)
compute_comm_view["communication KiB sent/rank"] = (
    compute_comm_view.pop("communication_bytes_sent_per_rank") / 1024
).round(3)
compute_comm_view["aggregate MiB sent/step"] = (
    compute_comm_view.pop("communication_bytes_sent_aggregate") / 1024**2
).round(3)
display(compute_comm_view)

output_dir = repository_root / "Assignments/S12"
plot_path = make_plots(comparison, output_dir)
comparison.to_csv(output_dir / "zero_comparison.csv", index=False)
from IPython.display import Image
display(Image(filename=str(plot_path)))

## My takeaways

- ZeRO does not make the model's matrix multiplications disappear. In this estimate every stage has the same forward/backward FLOPs; sharding changes memory placement and the collectives needed to keep training synchronized.
- The biggest memory win is stage 3 because it also partitions parameters. Aggregate storage falls as replicated state is removed: parameters and gradients are each kept once across the ranks under ZeRO-3, while optimizer state is also partitioned.
- The memory savings cost communication. In this small example, stages 1–3 can send more data per rank than the baseline, especially stage 3's parameter gathers. On a real large model, the reduced memory may be worth that cost; on a tiny MLP, it may not be.
- These numbers are analytic estimates for FP32 and ordinary Adam states. They do not include activation checkpointing, mixed precision, optimizer implementation details, temporary gathered weights, network latency, overlap, or hardware effects. No conclusion about real 32-GPU speed is possible from this CPU simulation.